In [0]:
spark.sql("""
CREATE OR REPLACE TABLE fraud_project.gold.fraud_by_day_of_week AS
SELECT day_of_week, COUNT(*) AS fraud_count
FROM fraud_project.silver.transactions
WHERE is_fraud = true
GROUP BY day_of_week
ORDER BY fraud_count DESC
""")
display(spark.table("fraud_project.gold.fraud_by_day_of_week"))

day_of_week,fraud_count
Sunday,2646
Friday,2284
Thursday,2082
Tuesday,2037
Monday,1747
Saturday,1434
Wednesday,1102


In [0]:
spark.sql("""
CREATE OR REPLACE TABLE fraud_project.gold.fraud_rate_trend AS
SELECT 
  date_trunc('day', date) AS txn_day,
  SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) AS fraud_count,
  COUNT(*) AS total_count,
  SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) / COUNT(*) AS fraud_rate
FROM fraud_project.silver.transactions
WHERE date >= (SELECT date_sub(MAX(date), 30) FROM fraud_project.silver.transactions)
GROUP BY 1 ORDER BY 1
""")
display(spark.table("fraud_project.gold.fraud_rate_trend"))

txn_day,fraud_count,total_count,fraud_rate
2019-10-01T00:00:00.000Z,8,3363,0.0023788284269997025
2019-10-02T00:00:00.000Z,0,3541,0.0
2019-10-03T00:00:00.000Z,10,3897,0.0025660764690787785
2019-10-04T00:00:00.000Z,11,3995,0.002753441802252816
2019-10-05T00:00:00.000Z,13,3862,0.0033661315380631796
2019-10-06T00:00:00.000Z,0,3973,0.0
2019-10-07T00:00:00.000Z,0,3906,0.0
2019-10-08T00:00:00.000Z,20,3478,0.005750431282346176
2019-10-09T00:00:00.000Z,0,3585,0.0
2019-10-10T00:00:00.000Z,10,3887,0.002572678157962439


In [0]:
spark.sql("""
CREATE OR REPLACE TABLE fraud_project.gold.top_fraud_users AS
SELECT client_id, COUNT(*) AS fraud_txn_count
FROM fraud_project.silver.transactions
WHERE is_fraud = true
GROUP BY client_id
ORDER BY fraud_txn_count DESC
""")
display(spark.table("fraud_project.gold.top_fraud_users"))

client_id,fraud_txn_count
1102,58
209,52
27,45
155,44
1128,43
1741,42
1851,42
989,42
1649,41
1926,39


In [0]:
spark.sql("""
CREATE OR REPLACE TABLE fraud_project.gold.spending_spikes AS
WITH weekly AS (
  SELECT client_id, date_trunc('week', date) AS wk, AVG(amount) AS avg_wk_amt
  FROM fraud_project.silver.transactions
  GROUP BY client_id, date_trunc('week', date)
)
SELECT t.client_id, t.transaction_id, t.date, t.amount, w.avg_wk_amt,
       t.amount / NULLIF(w.avg_wk_amt, 0) AS spike_ratio
FROM fraud_project.silver.transactions t
JOIN weekly w ON t.client_id = w.client_id AND date_trunc('week', t.date) = w.wk
WHERE t.amount > 3 * w.avg_wk_amt
""")
display(spark.table("fraud_project.gold.spending_spikes"))

client_id,transaction_id,date,amount,avg_wk_amt,spike_ratio
323,8427649,2010-08-24T21:16:00.000Z,74.0,16.38078947368421,4.5174867865118
1730,9562358,2011-05-21T05:43:00.000Z,136.78,19.150000000000002,7.142558746736292
932,11794290,2012-10-13T05:34:00.000Z,578.67,61.54968749999999,9.401672429287315
699,14299227,2014-04-20T13:16:00.000Z,8.46,-2.3726086956521724,-3.5656954370533285
50,17976165,2016-06-21T17:31:00.000Z,405.0,38.68105263157895,10.47024246877296
1315,18742224,2016-12-02T08:01:00.000Z,230.52,11.244117647058824,20.501386345801727
1830,20282751,2017-10-24T19:04:00.000Z,60.62,15.84294117647059,3.826309731556083
391,10032624,2011-09-06T20:51:00.000Z,620.62,113.9190909090909,5.447901620767531
45,12692895,2013-05-02T15:26:00.000Z,58.0,18.5,3.135135135135135
1495,15644271,2015-02-07T13:12:00.000Z,192.46,55.75212121212121,3.4520659629746393


In [0]:
spark.sql("""
CREATE OR REPLACE TABLE fraud_project.gold.fraud_by_mcc AS
SELECT mcc_description,
       COUNT(*) AS total_txn,
       SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) AS fraud_txn,
       SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) / COUNT(*) AS fraud_rate,
       SUM(CASE WHEN is_fraud THEN amount ELSE 0 END) AS total_fraud_amount
FROM fraud_project.silver.transactions
GROUP BY mcc_description
ORDER BY fraud_rate DESC
""")
display(spark.table("fraud_project.gold.fraud_by_mcc"))

mcc_description,total_txn,fraud_txn,fraud_rate,total_fraud_amount
Cruise Lines,428,165,0.3855140186915888,185946.78
Music Stores - Musical Instruments,319,76,0.23824451410658307,7562.26
Miscellaneous Fabricated Metal Products,351,29,0.08262108262108261,8543.36
"Computers, Computer Peripheral Equipment",2793,204,0.07303974221267455,25615.110000000004
Floor Covering Stores,334,23,0.0688622754491018,6643.92
Electronics Stores,6997,402,0.0574531942260969,61171.380000000005
Miscellaneous Metal Fabrication,391,22,0.056265984654731455,6943.780000000001
Fabricated Structural Metal Products,408,22,0.05392156862745098,6234.74
Precious Stones and Metals,5180,242,0.04671814671814672,26575.96
Coated and Laminated Products,381,17,0.04461942257217848,5642.57


In [0]:
spark.sql("""
CREATE OR REPLACE TABLE fraud_project.gold.fraud_by_merchant AS
SELECT merchant_id,
       COUNT(*) AS total_txn,
       SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) AS fraud_txn
FROM fraud_project.silver.transactions
GROUP BY merchant_id
ORDER BY fraud_txn DESC
""")
display(spark.table("fraud_project.gold.fraud_by_merchant"))

merchant_id,total_txn,fraud_txn
60569,301657,764
27092,589140,725
48919,86481,332
99370,34317,302
32858,66193,289
76639,5135,286
83018,1155,236
67570,35709,225
31893,22787,214
34490,45759,197


In [0]:
spark.sql("""
CREATE OR REPLACE TABLE fraud_project.gold.fraud_by_time_of_day AS
SELECT time_bucket, COUNT(*) AS fraud_count
FROM fraud_project.silver.transactions
WHERE is_fraud = true
GROUP BY time_bucket
""")
display(spark.table("fraud_project.gold.fraud_by_time_of_day"))

time_bucket,fraud_count
Night,418
Afternoon,5693
Evening,1480
Morning,5741


In [0]:
spark.sql("""
CREATE OR REPLACE TABLE fraud_project.gold.avg_amount_fraud_vs_legit AS
SELECT is_fraud, AVG(amount) AS avg_amount, COUNT(*) AS n
FROM fraud_project.silver.transactions
GROUP BY is_fraud
""")
display(spark.table("fraud_project.gold.avg_amount_fraud_vs_legit"))

is_fraud,avg_amount,n
true,110.23468196819688,13332
false,42.90858093569862,13292583


In [0]:
spark.sql("""
CREATE OR REPLACE TABLE fraud_project.gold.daily_fraud_losses AS
SELECT date_trunc('day', date) AS txn_day, SUM(amount) AS total_fraud_loss
FROM fraud_project.silver.transactions
WHERE is_fraud = true
GROUP BY 1 ORDER BY 1
""")
display(spark.table("fraud_project.gold.daily_fraud_losses"))

txn_day,total_fraud_loss
2010-01-01T00:00:00.000Z,0.19
2010-01-03T00:00:00.000Z,339.0
2010-01-04T00:00:00.000Z,11.64
2010-01-05T00:00:00.000Z,8.76
2010-01-07T00:00:00.000Z,-48.45999999999998
2010-01-08T00:00:00.000Z,383.24
2010-01-09T00:00:00.000Z,23.1
2010-01-10T00:00:00.000Z,530.01
2010-01-11T00:00:00.000Z,302.7
2010-01-12T00:00:00.000Z,1014.41


In [0]:
spark.sql("""
CREATE OR REPLACE TABLE fraud_project.gold.weekly_unique_fraud_users AS
SELECT date_trunc('week', date) AS wk, COUNT(DISTINCT client_id) AS unique_fraud_users
FROM fraud_project.silver.transactions
WHERE is_fraud = true
GROUP BY 1 ORDER BY 1
""")
display(spark.table("fraud_project.gold.weekly_unique_fraud_users"))

wk,unique_fraud_users
2009-12-28T00:00:00.000Z,1
2010-01-04T00:00:00.000Z,5
2010-01-11T00:00:00.000Z,5
2010-01-18T00:00:00.000Z,7
2010-01-25T00:00:00.000Z,17
2010-02-01T00:00:00.000Z,16
2010-02-08T00:00:00.000Z,12
2010-02-15T00:00:00.000Z,13
2010-02-22T00:00:00.000Z,13
2010-03-01T00:00:00.000Z,10


In [0]:
spark.sql("""
CREATE OR REPLACE TABLE fraud_project.gold.monthly_fraud_seasonality AS
SELECT date_trunc('month', date) AS mth, COUNT(*) AS fraud_count
FROM fraud_project.silver.transactions
WHERE is_fraud = true
GROUP BY 1 ORDER BY 1
""")
display(spark.table("fraud_project.gold.monthly_fraud_seasonality"))

mth,fraud_count
2010-01-01T00:00:00.000Z,107
2010-02-01T00:00:00.000Z,259
2010-03-01T00:00:00.000Z,261
2010-04-01T00:00:00.000Z,237
2010-05-01T00:00:00.000Z,274
2010-06-01T00:00:00.000Z,182
2010-07-01T00:00:00.000Z,244
2010-08-01T00:00:00.000Z,229
2010-09-01T00:00:00.000Z,193
2010-10-01T00:00:00.000Z,224


In [0]:
spark.sql("""
CREATE OR REPLACE TABLE fraud_project.gold.behavior_around_fraud_event AS
WITH fraud_events AS (
  SELECT client_id, date AS fraud_date
  FROM fraud_project.silver.transactions
  WHERE is_fraud = true
)
SELECT f.client_id, f.fraud_date,
   AVG(CASE WHEN t.date BETWEEN date_sub(f.fraud_date, 7) AND f.fraud_date THEN t.amount END) AS avg_before,
   AVG(CASE WHEN t.date BETWEEN f.fraud_date AND date_add(f.fraud_date, 7) THEN t.amount END) AS avg_after
FROM fraud_events f
JOIN fraud_project.silver.transactions t ON t.client_id = f.client_id
GROUP BY f.client_id, f.fraud_date
""")
display(spark.table("fraud_project.gold.behavior_around_fraud_event"))

client_id,fraud_date,avg_before,avg_after
1963,2019-04-25T18:10:00.000Z,34.20462962962962,36.601176470588236
115,2019-07-30T19:31:00.000Z,56.27600000000001,60.159375000000004
1872,2016-11-17T11:32:00.000Z,88.07666666666667,88.70999999999998
1789,2015-05-29T10:09:00.000Z,50.973,87.65695652173913
819,2019-05-11T12:48:00.000Z,79.5952380952381,26.837083333333336
963,2016-10-04T11:16:00.000Z,30.10636363636364,21.88888888888889
1963,2010-02-02T05:58:00.000Z,46.655,28.533214285714287
115,2019-08-02T11:23:00.000Z,57.76652173913045,65.06962962962962
1872,2016-11-17T12:26:00.000Z,85.64684210526316,75.79099999999998
1789,2015-05-29T14:26:00.000Z,71.26227272727273,82.63238095238096


In [0]:
spark.sql("""
CREATE OR REPLACE TABLE fraud_project.gold.fraud_by_amount_tier AS
SELECT CASE WHEN amount < 50 THEN 'Low' WHEN amount < 500 THEN 'Medium' ELSE 'High' END AS amount_tier,
       SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) / COUNT(*) AS fraud_rate,
       COUNT(*) AS total_txn
FROM fraud_project.silver.transactions
GROUP BY 1
""")
display(spark.table("fraud_project.gold.fraud_by_amount_tier"))

amount_tier,fraud_rate,total_txn
Medium,0.0017086665742695876,4415724
Low,6.138924158910308E-4,8846827
High,0.008209574762475786,43364


In [0]:
gold_tables = [
    "fraud_by_day_of_week", "fraud_rate_trend", "top_fraud_users", "spending_spikes",
    "fraud_by_mcc", "fraud_by_merchant", "fraud_by_time_of_day", "avg_amount_fraud_vs_legit",
    "daily_fraud_losses", "weekly_unique_fraud_users", "monthly_fraud_seasonality",
    "behavior_around_fraud_event", "fraud_by_amount_tier"
]

for t in gold_tables:
    cnt = spark.sql(f"SELECT COUNT(*) as cnt FROM fraud_project.gold.{t}").collect()[0]["cnt"]
    print(f"{t}: {cnt} rows")

fraud_by_day_of_week: 7 rows
fraud_rate_trend: 31 rows
top_fraud_users: 1196 rows
spending_spikes: 1056488 rows
fraud_by_mcc: 108 rows
fraud_by_merchant: 74831 rows
fraud_by_time_of_day: 4 rows
avg_amount_fraud_vs_legit: 2 rows
daily_fraud_losses: 1571 rows
weekly_unique_fraud_users: 326 rows
monthly_fraud_seasonality: 79 rows
behavior_around_fraud_event: 13265 rows
fraud_by_amount_tier: 3 rows
